In [20]:
import pandas as pd
import os
import re
import spacy
from gender_classifier import GenderClassifier
from gender_rewrite import GenderRewrite
from tqdm import tqdm
import json


In [21]:
pd.set_option('display.max_colwidth', None)


In [22]:
data_dir = "./data"
raw_dir = os.path.join(data_dir, "structured", "situations")
processed_dir = os.path.join(data_dir, "processed")

situation_number = 1

situation_filename = f"situation_{situation_number}.csv"
raw_path = os.path.join(raw_dir, situation_filename)
processed_path = os.path.join(processed_dir, situation_filename)


In [23]:
nlp = spacy.load("es_core_news_sm")


In [24]:
df = pd.read_csv(raw_path)


In [25]:
with open("./metadata.json", "r", encoding="utf-8") as f:
    metadata = json.load(f)
    

In [26]:
choices_map = metadata.get(str(situation_number), {}).get("choices", {})

df["choice"] = df["choice"].map(choices_map).fillna(df["choice"])


In [27]:
remove_texts = metadata.get(str(situation_number), {}).get("remove", [])

mask_remove = df["text"].isin(remove_texts)

removed_rows = df[mask_remove]

df = df[~mask_remove]

display(removed_rows)


,timestamp,age,gender,sexual_orientation,choice,text
13,4/05/2026 20:16:23,21,Masculino,Heterosexual,friendly,"Baseado en mi personalidad, si una persuna fueste cordial y amigable comigo, yo sería amigable y cordial con la persona."
22,8/05/2026 20:20:30,57,Femenino,Heterosexual,friendly,Educada


In [28]:
WHITELIST_NAMES = {
	"Laura"
}

NAME_REPLACEMENTS = {
	"María"
}

REPLACEMENT = "{{name}}"

def repl(m):
	intro = m.group(0)
	nombre = m.group(1)

	return intro.replace(nombre, REPLACEMENT)

def anonymize_names(text):
	if pd.isna(text):
		return text

	out = str(text)

	out = re.sub(r"\s+", " ", out).strip()

	for name in WHITELIST_NAMES:
		out = re.sub(rf"\b{name}\b", f"__NAME_{name}__", out)

	patterns = [
		r"\bsoy\s+(mi nombre|me llamo|\.\.\.|[^\s,;:.…!?]+)",
		r"\bme llamo\s+([^\s,;:.…!?]+)",
	]

	for pat in patterns:
		out = re.sub(pat, repl, out, flags=re.IGNORECASE)

	out = re.sub(
		rf"\bsoy\b(?!\s*{re.escape(REPLACEMENT)})",
		f"soy {REPLACEMENT}",
		out,
		flags=re.IGNORECASE
	)

	for name in NAME_REPLACEMENTS:
		out = re.sub(rf"\b{name}\b", lambda m: REPLACEMENT, out)

	for name in WHITELIST_NAMES:
		out = out.replace(f"__NAME_{name}__", name)
		
	out = re.sub(r"\s+([,;:.!?])", r"\1", out)

	return out



In [29]:
df = df[df["text"].notna()].copy()

df["text_clean"] = df["text"].apply(anonymize_names)

display(df)


,timestamp,age,gender,sexual_orientation,choice,text,text_clean
0,18/04/2026 21:20:28,21,Femenino,Heterosexual,friendly,"Hola, soy Patri! Encantada de conocerte","Hola, soy {{name}}! Encantada de conocerte"
1,19/04/2026 15:15:37,24,Femenino,Homosexual,friendly,"Holaaa buenas, soy Nombre, acabo de llegar y estoy todavía un poco perdida, y tú?","Holaaa buenas, soy {{name}}, acabo de llegar y estoy todavía un poco perdida, y tú?"
2,19/04/2026 15:44:41,23,Otros,Homosexual,shy,"Hola, Laura, encantado.","Hola, Laura, encantado."
3,19/04/2026 20:57:37,20,Otros,Homosexual,shy,Emmm. Hola.,Emmm. Hola.
4,20/04/2026 15:15:28,23,Femenino,Homosexual,friendly,Hola Laura yo soy Rocío encantada,Hola Laura yo soy {{name}} encantada
5,21/04/2026 15:19:26,24,Femenino,Bisexual,friendly,"Hol, soy Briana un gusto","Hol, soy {{name}} un gusto"
6,4/05/2026 17:19:20,23,Masculino,Heterosexual,shy,Hola,Hola
7,4/05/2026 18:35:52,18,Masculino,Heterosexual,friendly,"Buenas, yo soy Miguel, encantado","Buenas, yo soy {{name}}, encantado"
8,4/05/2026 18:38:38,18,Masculino,Heterosexual,friendly,Encantado soy ...,Encantado soy {{name}}
9,4/05/2026 19:26:02,22,Masculino,Heterosexual,friendly,"Hola, soy Gianfranco un placer","Hola, soy {{name}} un placer"


In [30]:
before = len(df)

subset_cols = ["text_clean"]

if "choice" in df.columns:
    subset_cols.append("choice")

df = df.drop_duplicates(
	subset=subset_cols,
	keep="first"
)

after = len(df)

print(f"Filas eliminadas: {before - after}")
print(f"Filas restantes: {after}")


Filas eliminadas: 1
Filas restantes: 25


In [31]:
clf = GenderClassifier()

text = "Hol, soy {{name}} un gusto."

result = clf.predict(text)

clf.print_prediction(text)


Predicted class: neutral
Confidence: 0.9669

Confidence scores:
male: 0.0220
female: 0.0111
neutral: 0.9669


In [32]:
chain = GenderRewrite()

# text = "Hola, soy {{name}}! Encantada de conocerte"

# result = chain.rewrite(
# 	text,
# 	source_gender="femenino",
# 	target_gender="masculino"
# )

# print(result)


In [33]:
text_clean = df["text_clean"].to_list()
texts = df["text"].to_list()

clf_results = clf.predict_batch(text_clean, return_all_scores=True)

NEUTRAL_LABEL = "neutral"
THRESHOLD = 0.98

gender_map = {
	"male": "masculino",
	"female": "femenino",
}

opposite_gender_map = {
	"masculino": "femenino",
	"femenino": "masculino",
}

flipped_texts = []
flip_flags = []
confidences = []

for text, res in tqdm(zip(text_clean, clf_results)):
	all_scores = res["all_scores"]

	best_label = res["label"]
	best_conf = res["confidence"]

	should_flip = best_label != NEUTRAL_LABEL and best_conf >= THRESHOLD

	if should_flip:
		detected_gender = gender_map[best_label]
		target_gender = opposite_gender_map[detected_gender]

		flipped = chain.rewrite(
			text,
			source_gender=detected_gender,
			target_gender=target_gender
		)
	else:
		flipped = text

	flipped_texts.append(flipped)
	flip_flags.append(should_flip)
	confidences.append(best_conf)

df["text_flipped"] = flipped_texts
df["was_flipped"] = flip_flags
df["gender_confidence"] = confidences


25it [00:29,  1.18s/it]


In [34]:
display(df)

,timestamp,age,gender,sexual_orientation,choice,text,text_clean,text_flipped,was_flipped,gender_confidence
0,18/04/2026 21:20:28,21,Femenino,Heterosexual,friendly,"Hola, soy Patri! Encantada de conocerte","Hola, soy {{name}}! Encantada de conocerte","Hola, soy {{name}}! Encantado de conocerte.",True,0.998070
1,19/04/2026 15:15:37,24,Femenino,Homosexual,friendly,"Holaaa buenas, soy Nombre, acabo de llegar y estoy todavía un poco perdida, y tú?","Holaaa buenas, soy {{name}}, acabo de llegar y estoy todavía un poco perdida, y tú?","Holaaa buenas, soy {{name}}, acabo de llegar y estoy todavía un poco perdido, y tú?",True,0.996565
2,19/04/2026 15:44:41,23,Otros,Homosexual,shy,"Hola, Laura, encantado.","Hola, Laura, encantado.","Hola, Laura, encantada.",True,0.999179
3,19/04/2026 20:57:37,20,Otros,Homosexual,shy,Emmm. Hola.,Emmm. Hola.,Emmm. Hola.,False,0.993859
4,20/04/2026 15:15:28,23,Femenino,Homosexual,friendly,Hola Laura yo soy Rocío encantada,Hola Laura yo soy {{name}} encantada,Hola Laura yo soy él encantado,True,0.999370
5,21/04/2026 15:19:26,24,Femenino,Bisexual,friendly,"Hol, soy Briana un gusto","Hol, soy {{name}} un gusto","Hol, soy {{name}} un gusto",False,0.980689
6,4/05/2026 17:19:20,23,Masculino,Heterosexual,shy,Hola,Hola,Hola,False,0.998551
7,4/05/2026 18:35:52,18,Masculino,Heterosexual,friendly,"Buenas, yo soy Miguel, encantado","Buenas, yo soy {{name}}, encantado","Buenas, yo soy {{name}}, encantado",False,0.847326
8,4/05/2026 18:38:38,18,Masculino,Heterosexual,friendly,Encantado soy ...,Encantado soy {{name}},Encantado soy {{name}},False,0.876114
9,4/05/2026 19:26:02,22,Masculino,Heterosexual,friendly,"Hola, soy Gianfranco un placer","Hola, soy {{name}} un placer","Hola, soy {{name}} un placer",False,0.993907


In [35]:
df["text_flipped"] = df["text_flipped"].apply(anonymize_names)



In [36]:
display(df[["text", "text_flipped", "was_flipped", "gender_confidence"]])


,text,text_flipped,was_flipped,gender_confidence
0,"Hola, soy Patri! Encantada de conocerte","Hola, soy {{name}}! Encantado de conocerte.",True,0.998070
1,"Holaaa buenas, soy Nombre, acabo de llegar y estoy todavía un poco perdida, y tú?","Holaaa buenas, soy {{name}}, acabo de llegar y estoy todavía un poco perdido, y tú?",True,0.996565
2,"Hola, Laura, encantado.","Hola, Laura, encantada.",True,0.999179
3,Emmm. Hola.,Emmm. Hola.,False,0.993859
4,Hola Laura yo soy Rocío encantada,Hola Laura yo soy {{name}} encantado,True,0.999370
5,"Hol, soy Briana un gusto","Hol, soy {{name}} un gusto",False,0.980689
6,Hola,Hola,False,0.998551
7,"Buenas, yo soy Miguel, encantado","Buenas, yo soy {{name}}, encantado",False,0.847326
8,Encantado soy ...,Encantado soy {{name}},False,0.876114
9,"Hola, soy Gianfranco un placer","Hola, soy {{name}} un placer",False,0.993907


In [37]:
os.makedirs(processed_dir, exist_ok=True)

df.to_csv(processed_path, index=False)

